In [ ]:
!pip install -q transformers datasets accelerate peft trl bitsandbytes

In [ ]:
# ---------------- 1. KÜTÜPHANELER ----------------
import gc
import re
import time

import numpy as np
import pandas as pd
import torch

import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

from huggingface_hub import (
    login,
    HfApi,
    create_repo,
    snapshot_download,
    hf_hub_download
)

from transformers.trainer_utils import get_last_checkpoint

In [ ]:
# ---------------- 2. HUGGING FACE GİRİŞİ ----------------
from google.colab import userdata
hf_token = userdata.get("hf")
login(token=hf_token)

In [ ]:
# ---------------- 3. VERİ SETİ YÜKLEME VE HAZIRLAMA ----------------
print("\nVeri seti Hugging Face Hub üzerinden indiriliyor...")

dataset_path = hf_hub_download(
    repo_id="nihalenc/turkish-8class-emotion-dataset",
    filename="turkish-8class-emotion-dataset.csv",
    repo_type="dataset"
)

df = pd.read_csv(dataset_path, sep=";", encoding="utf-8")
df.columns = df.columns.str.strip()

# ---------------- METİN TEMİZLİĞİ ----------------
def clean_text_for_llm(text):
    text = str(text)
    # URL'leri siler
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Kullanıcı mentionları silinir.
    text = re.sub(r'\@\w+', '', text)
    # HTML etiketlerini siler
    text = re.sub(r'<.*?>', '', text)
    # Baştaki/sondaki boşlukları temizler.
    return text.strip()

# Temizleme fonksiyonunu tüm text sütununa uygular.
df['text'] = df['text'].apply(clean_text_for_llm)

# Aynı metinlerin birden fazla kez bulunmasını engeller.
print(f"Kopya kontrolü öncesi: {len(df)} satır")
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
print(f"Kopya kontrolü sonrası: {len(df)} satır")

emotion_cols = [
    'igrenme',
    'korku',
    'minnet',
    'pismanlik',
    'mutluluk',
    'ofke',
    'saskinlik',
    'uzuntu'
]

# Her satırda 1 olan duygu sütununu bulup etiket adını çıkarır.
df['label_name'] = df[emotion_cols].idxmax(axis=1)

# Duygu adlarını sayısal etiketlere çevirir.
label_mapping = {col: idx for idx, col in enumerate(emotion_cols)}
df['label'] = df['label_name'].map(label_mapping)
print("Yeni veri boyutu:", len(df))

# Her sınıfta kaç örnek olduğunu gösterir.
print(df["label_name"].value_counts())

# ---------------- TRAIN / VALIDATION / TEST AYIRMA ----------------
X = df['text'].tolist()
y = df['label'].tolist()

# %80 eğitim, %20 geçici set ayrılır.
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Geçici set ikiye ayrılır: %10 validation, %10 test.
# stratify=y_temp sınıf oranlarının val/test içinde de korunmasını sağlar.
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print(f"\nDağılım -> Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"Eğitim Seti (Train): {len(X_train)} satır")
print(f"Doğrulama Seti (Val): {len(X_val)} satır")
print(f"Test Seti (Test): {len(X_test)} satır")

train_dataset = Dataset.from_dict({'text': X_train, 'label': y_train})
val_dataset = Dataset.from_dict({'text': X_val, 'label': y_val})
test_dataset = Dataset.from_dict({'text': X_test, 'label': y_test})

In [ ]:
# ---------------- 4. SINIF AĞIRLIKLARI ----------------
class_weights = compute_class_weight(
    class_weight='balanced',      # Sınıfları örnek sayılarına göre dengeler.
    classes=np.unique(y_train),
    y=y_train                    
)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
print("\nSınıf Ağırlıkları :", class_weights)

In [ ]:
# ---------------- 5. LLAMA-3 MODEL, TOKENIZER, QUANTIZATION VE LORA ----------------
torch.cuda.empty_cache()
gc.collect()

# Kullanılacak temel model
model_id = "meta-llama/Meta-Llama-3-8B"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    add_eos_token=True
)

tokenizer.pad_token = tokenizer.eos_token

# ---------------- 4-BIT QUANTIZATION AYARLARI ----------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,               
    bnb_4bit_use_double_quant=True,   
    bnb_4bit_quant_type="nf4",        
    bnb_4bit_compute_dtype=torch.float16 
)

# Sequence classification modeli yüklenir.
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=8,
    quantization_config=bnb_config,
    device_map="auto" 
)

model.config.pad_token_id = tokenizer.pad_token_id

# Bazı katmanları ve hesaplamaları k-bit fine-tuning için hazırlar.
model = prepare_model_for_kbit_training(model)

# ---------------- LORA AYARLARI ----------------
peft_config = LoraConfig(
    task_type="SEQ_CLS",      # Görev türü: sequence classification yani metin sınıflandırma.
    r=16,                    
    lora_alpha=32,            
    lora_dropout=0.05,      
    bias="none",              
    target_modules=[
        "q_proj",          
        "k_proj",         
        "v_proj",         
        "o_proj",            
        "gate_proj",          
        "up_proj",          
        "down_proj"      
    ],
    modules_to_save=["score"] 
)

# LoRA katmanlarını temel modele ekler.
model = get_peft_model(model, peft_config)

In [ ]:
# ---------------- 6. TOKENIZATION ----------------
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Batch içindeki metinleri dinamik olarak aynı uzunluğa tamamlar.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# ---------------- 7. SINIF AĞIRLIKLI TRAINER ----------------
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = torch.nn.CrossEntropyLoss(
            weight=class_weights_tensor.to(logits.device)
        )

        loss = loss_fct(
            logits.view(-1, self.model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

# ---------------- DEĞERLENDİRME METRİKLERİ ----------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        'accuracy': accuracy_score(labels, predictions),
        'macro_f1': f1_score(labels, predictions, average='macro'),
        'macro_precision': precision_score(labels, predictions, average='macro', zero_division=0),
        'macro_recall': recall_score(labels, predictions, average='macro', zero_division=0)
    }

# ---------------- TRAINING ARGUMENTS ----------------
training_args = TrainingArguments(
    output_dir="./llama3-emotion-results",  

    learning_rate=1e-4,  

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,  
    gradient_accumulation_steps=4,  

    gradient_checkpointing=False, 
    num_train_epochs=3,            
    lr_scheduler_type="cosine",  
    warmup_steps=150,            

    eval_strategy="steps",  
    eval_steps=250,         
    save_strategy="steps", 
    save_steps=250,        
    save_total_limit=1,    

    load_best_model_at_end=True,     # Eğitim sonunda en iyi validation sonucuna sahip model geri yüklenir.
    metric_for_best_model="macro_f1",# En iyi model seçimi macro_f1 değerine göre yapılır.
    greater_is_better=True,          # macro_f1 yükseldikçe model daha iyi kabul edilir.

    logging_steps=100, 

    fp16=False, 
    bf16=True, 

    optim="paged_adamw_8bit", 
    dataloader_num_workers=2, 

    report_to="none", 
    seed=42,           
    data_seed=42,      
    weight_decay=0.01, 
    push_to_hub=False,                        
)

# ---------------- TRAINER BAŞLATMA ----------------
trainer = WeightedTrainer(
    model=model,                  
    args=training_args,          
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,  
    data_collator=data_collator, 
    compute_metrics=compute_metrics, 

    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3),
    ]
)

In [ ]:
# ---------------- 8. CHECKPOINT'TEN DEVAM ETME, EĞİTİM VE HUGGING FACE'E YÜKLEME ----------------
repo_id = "nihalenc/llama3-duygu-analizi-lora"
local_output_dir = "./llama3-emotion-results"
api = HfApi(token=hf_token)
create_repo(
    repo_id=repo_id,
    repo_type="model",
    private=True,
    exist_ok=True,
    token=hf_token
)

print("Hugging Face Hub üzerinde eski checkpoint kontrol ediliyor...")

try:
    # Daha önce Hub'a yüklenmiş checkpoint dosyaları varsa yerel klasöre indirilir.
    # Böylece eğitim yarıda kaldıysa kaldığı yerden devam edebilir.
    snapshot_download(
        repo_id=repo_id,
        repo_type="model",
        local_dir=local_output_dir,
        allow_patterns=["checkpoint-*/*"],
        token=hf_token
    )

    print("Hub üzerindeki checkpoint dosyaları yerel klasöre indirildi.")

except Exception as e:
    # Repo boşsa veya checkpoint yoksa eğitim sıfırdan başlar.
    print("Hub üzerinde eski checkpoint bulunamadı. Eğitim sıfırdan başlayacak.")

# Yerel klasördeki en son checkpoint bulunur.
last_checkpoint = get_last_checkpoint(local_output_dir)

# Checkpoint varsa eğitim kaldığı yerden devam eder.
# Checkpoint yoksa eğitim sıfırdan başlar.
if last_checkpoint is not None:
    print(f"Eğitim {last_checkpoint} checkpoint'inden devam ediyor...")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Checkpoint bulunamadı. Eğitim sıfırdan başlıyor...")
    trainer.train()


id2label = {i: label for i, label in enumerate(emotion_cols)}
label2id = {label: i for i, label in id2label.items()}

trainer.model.config.id2label = id2label
trainer.model.config.label2id = label2id

trainer.save_model(local_output_dir)
tokenizer.save_pretrained(local_output_dir)


print("Final model Hugging Face Hub'a yükleniyor...")

# Yerel klasördeki model, tokenizer, config ve checkpoint dosyaları Hugging Face repo'suna yüklenir.
api.upload_folder(
    folder_path=local_output_dir,
    repo_id=repo_id,
    repo_type="model",
    commit_message="Final LLaMA-3 duygu analizi modeli",
    token=hf_token
)

print("Final model, tokenizer ve checkpoint dosyaları Hugging Face Hub'a başarıyla yüklendi.")

In [ ]:
# ---------------- 9. TEST TAHMİNLERİ, CONFUSION MATRIX VE HATA ANALİZİ ----------------

print("Model test seti üzerinde tahmin yapıyor..")

# Test setinden tahminler alınır.
predictions = trainer.predict(tokenized_test)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

print("\n" + "="*70)
print("GENEL TEST METRİKLERİ")
print("="*70)
print(f"Test Sonuçları: {predictions.metrics}\n")


# ---------------- CONFUSION MATRIX ----------------
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))

# Heatmap, modelin hangi sınıfları hangi sınıflarla karıştırdığını görselleştirir.
sns.heatmap(
    cm,
    annot=True,          
    fmt='d',           
    cmap='Blues',       
    xticklabels=emotion_cols,
    yticklabels=emotion_cols
)

plt.xlabel('Modelin Tahmin Ettiği Duygu', fontsize=12, fontweight='bold')
plt.ylabel('Gerçek Duygu', fontsize=12, fontweight='bold')
plt.title('Llama-3 Duygu Analizi Karmaşıklık Matrisi', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# ---------------- CLASSIFICATION REPORT ----------------
print("\n" + "="*70)
print("HER DUYGU SINIFI İÇİN BAŞARI YÜZDELERİ (Classification Report)")
print("="*70)

# Her sınıf için precision, recall, f1-score ve support değerlerini verir.
print(classification_report(y_true, y_pred, target_names=emotion_cols))

# ---------------- DETAYLI HATA KOMBİNASYONLARI ----------------
print("\n" + "="*70)
print("MODELİN TÜM HATA KOMBİNASYONLARI (En Çok Karıştırılandan En Aza)")
print("="*70)

hatalar = []
for gercek_idx in range(len(emotion_cols)):
    for tahmin_idx in range(len(emotion_cols)):
        if gercek_idx != tahmin_idx:
            hata_sayisi = cm[gercek_idx, tahmin_idx]

            if hata_sayisi > 0:
                gercek_duygu = emotion_cols[gercek_idx].upper()
                tahmin_duygu = emotion_cols[tahmin_idx].upper()
                hatalar.append((gercek_duygu, tahmin_duygu, hata_sayisi))

# Hataları en çoktan en aza sıralar.
hatalar.sort(key=lambda x: x[2], reverse=True)

# Hangi gerçek sınıfın hangi tahmine dönüştüğünü yazdırır.
for gercek, tahmin, sayi in hatalar:
    print(f"Gerçekte {gercek:<10} olup modelin {tahmin:<10} dediği: {sayi:>3} cümle")

In [ ]:
# ---------------- 10. CLASSIFICATION REPORT'U TABLO VE CSV OLARAK KAYDETME ----------------

# output_dict=True sonucu sözlük formatında verir.
report = classification_report(
    y_true,
    y_pred,
    target_names=emotion_cols,
    output_dict=True,
    zero_division=0  
)

# Raporu tablo formatına çevirir.
report_df = pd.DataFrame(report).transpose()
display(report_df)
report_df.to_csv("llama3_classification_report.csv")

In [ ]:
# ---------------- 11. YANLIŞ TAHMİNLERİ CSV OLARAK ÇIKARMA ----------------
test_texts = X_test

# Her test örneği için metin, gerçek etiket ve model tahmini tabloya yazılır.
error_df = pd.DataFrame({
    "text": test_texts,
    "true_label": [emotion_cols[i] for i in y_true],
    "predicted_label": [emotion_cols[i] for i in y_pred]
})

# Sadece modelin yanlış tahmin ettiği örnekleri filtreler.
wrong_df = error_df[error_df["true_label"] != error_df["predicted_label"]]
display(wrong_df.head(30))

# Hatalı tahminleri CSV dosyasına kaydeder.
wrong_df.to_csv("llama3_wrong_predictions.csv", index=False)

print("Yanlış tahmin sayısı:", len(wrong_df))

In [ ]:
# ---------------- 12. INFERENCE TIME ÖLÇME ----------------
def measure_inference_time(texts, n=100):
    model.eval()
    sample_texts = texts[:n]
    times = []

    for text in sample_texts:
        start = time.time()

        # Metin tokenize edilir ve modelin bulunduğu cihaza taşınır.
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,     
            padding=True,    
            max_length=128      
        ).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs)
        end = time.time()

        # Cümlenin tahmin süresi listeye eklenir.
        times.append(end - start)

    # Ortalama saniye/cümle süresi hesaplanır.
    avg_time = np.mean(times)
    return avg_time

# İlk 100 test cümlesi için ortalama inference süresi ölçülür.
avg_inf_time = measure_inference_time(X_test, n=100)

print(f"Ortalama inference süresi: {avg_inf_time:.4f} saniye/cümle")
